#TODO Ainda é necessario tratar os horarios de chamados e tempo de atendimento.

In [0]:
from config import BRONZE_SCHEMA,SILVER_SCHEMA, CATALOG
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
bronze_chamados = spark.table(f"{BRONZE_SCHEMA}.chamados")
bronze_chamados.printSchema()
display(bronze_chamados)

In [0]:
chamados_sem_duplicados = bronze_chamados.drop('ingestion_datetime').dropDuplicates()
_, passou = dq_check_unique('chamados_silver', 'id_chamado único', chamados_sem_duplicados, ['id_chamado'])
if not passou:
    raise ValueError("Existem registros diferentes para o mesmo id_chamado.")
_, passou = dq_check('chamados_silver', 'id_chamado não nulo', chamados_sem_duplicados, F.col('id_chamado').isNotNull())
if not passou:
    raise ValueError("Existem chamados sem id_chamado.")

In [0]:
silver_chamados= (
    chamados_sem_duplicados 
    .withColumn(
        'nome_motivo',
        F.lower(F.expr("trim(motivo) COLLATE UNICODE_CI_AI"
        )),
    )
    .drop('motivo')
)
display(silver_chamados.groupBy('nome_motivo').count().orderBy('count'))

In [0]:
# Associa o motivo ao cadastro antes de remover sua descrição.
silver_motivos = spark.table(f"{SILVER_SCHEMA}.base_motivos")
motivos = (
    silver_motivos.select('id_motivo', 'nome_motivo')
    .withColumn('nome_motivo', F.expr("trim(nome_motivo) COLLATE UNICODE_CI_AI"))
)
_, passou = dq_check_unique('motivos_silver', 'nome_motivo único no join', motivos, ['nome_motivo'])
if not passou:
    raise ValueError("Existem motivos equivalentes no cadastro; o join pode duplicar chamados.")

silver_chamados = silver_chamados.join(motivos, on='nome_motivo', how='left')
_, passou = dq_check('chamados_silver', 'motivo encontrado', silver_chamados, F.col('id_motivo').isNotNull())
if not passou:
    raise ValueError("Existem chamados sem id_motivo; confira nome_motivo antes de continuar.")
silver_chamados = silver_chamados.drop('nome_motivo')
display(silver_chamados)

In [0]:
subs={
    '_': ' ',
    '.': '',
    'atend': 'atendimento',
    'chat bot': 'chatbot'
}
parciais =['atendimento']
def string_mapping(df,column,subs:dict,parciais:list):
    import re

    df_mapped = (
        df.withColumn(column, F.lower(F.trim(F.col(f'{column}'))),
                      )
    )
    expressao = F.col(column)
    for original, substituto in subs.items():
        expressao= F.regexp_replace(
            expressao, re.escape(original), substituto
        )
    df_mapped = df_mapped.withColumn(column, expressao)
    for parcial in parciais:
        df_mapped = df_mapped.withColumn(column,
                                     F.regexp_replace(
                                         F.col(column),
                                         rf"{parcial}\S*",
                                         parcial
                                     ))
    return df_mapped

df_ch = string_mapping(silver_chamados,'canal',subs,parciais)
display(df_ch.groupBy('canal').count())


In [0]:
silver_chamados = df_ch

In [0]:
# Mapeia valores completos e preserva nulos originais.
resolvido_por_valor = {
    'sim': 1, 's': 1, '1': 1,
    'não': 0, 'nao': 0, 'n': 0,
    'não resolvido': 0, 'nao resolvido': 0, '0': 0, '00': 0,
}
mapeamento = F.create_map(*[
    F.lit(valor) for par in resolvido_por_valor.items() for valor in par
])
resolvido_convertido = F.element_at(mapeamento, F.lower(F.trim(F.col('resolvido'))))
_, passou = dq_check(
    'chamados_silver', 'resolvido reconhecido', silver_chamados,
    F.col('resolvido').isNull() | resolvido_convertido.isNotNull(),
)
if not passou:
    raise ValueError("Existem valores preenchidos de resolvido fora do mapeamento.")
silver_chamados = silver_chamados.withColumn('resolvido', resolvido_convertido)
display(silver_chamados.groupBy('resolvido').count())

In [0]:
(silver_chamados.groupBy('hora_finalizacao_atendimento').count()).show()
(silver_chamados.groupBy('hora_abertura_chamado').count()).show()
silver_chamados = silver_chamados.drop('hora_abertura_chamado','hora_finalizacao_atendimento')


In [0]:
silver_chamados = silver_chamados.withColumn(
    "tempo_espera",
    F.expr("try_cast(tempo_espera AS DOUBLE)"),
)
silver_chamados = silver_chamados.withColumn(
    "tempo_atendimento",
    F.expr("try_cast(tempo_atendimento AS DOUBLE)"),
)
silver_chamados = silver_chamados.withColumn(
    'tempo_espera',
    F.when(F.col('hora_inicio_atendimento') =='igual a hora de abertura',F.lit(0)).otherwise(F.col('tempo_espera'))
)
silver_chamados= silver_chamados.drop('hora_inicio_atendimento')
display(silver_chamados)

In [0]:
#silver_chamados_hora = spark.table(f"{SILVER_SCHEMA}.chamados_hora")